# 04 · Results Dashboard

Aggregate and visualize results that already exist under `results/` —
no recomputation. Good for reading off the numbers that go into the paper.

In [ ]:
import nb_config as C
C.setup()
import json, os
import pandas as pd
import matplotlib.pyplot as plt

: 

## All reports found

In [ ]:
reports = C.find_reports()
for r in reports:
    print("  ", os.path.relpath(r, C.PROJECT_ROOT))
print(f"\n{len(reports)} report(s).")

: 

## Sample-size stability (coefficient of variation)
The headline robustness claim: M3 should have far lower CV than FID at small N.

In [ ]:
p = os.path.join(C.RESULTS_DIR, "n_scaling", "n_scaling_report.json")
if os.path.exists(p):
    d = C.load_json(p)
    rows = []
    for n, metrics in d["summary"].items():
        row = {"N": int(n)}
        for m, stat in metrics.items():
            row[f"{m}_cv"] = stat.get("cv")
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("N").set_index("N")
    display(df)
    df.plot(marker="o", figsize=(8, 4), title="Coefficient of variation vs N (lower = more stable)")
    plt.ylabel("CV (%)"); plt.tight_layout(); plt.show()
else:
    print("Run experiments/n_scaling.py first (see notebook 03).")

: 

## Noise-quality ladder
Each metric vs. injected Gaussian sigma. Monotonic, well-spread curves are good.
Normalized so all metrics share one axis.

In [ ]:
p = os.path.join(C.RESULTS_DIR, "noise_quality_ladder", "results.json")
if os.path.exists(p):
    rows = C.load_json(p)
    df = pd.DataFrame(rows)
    display(df)
    plt.figure(figsize=(8, 4))
    for col in ["m3", "fid", "kid", "cmmd"]:
        if col in df:
            v = df[col].astype(float)
            norm = (v - v.min()) / (v.max() - v.min() + 1e-9)
            plt.plot(df["sigma"], norm, marker="o", label=col)
    plt.xlabel("noise sigma"); plt.ylabel("normalized metric")
    plt.legend(); plt.title("Metric response to noise (normalized)")
    plt.tight_layout(); plt.show()
else:
    print("Run experiments/noise_quality_ladder.py first.")

: 

## OOD / anomaly detection AUC
Key claim: M3 separates anomalies better than FID.

In [ ]:
import glob
cands = glob.glob(os.path.join(C.RESULTS_DIR, "**", "*ood*", "**", "*.json"), recursive=True)
cands += glob.glob(os.path.join(C.RESULTS_DIR, "**", "*ood*.json"), recursive=True)
for c in sorted(set(cands)):
    try:
        d = C.load_json(c)
        flat = {k: v for k, v in (d.items() if isinstance(d, dict) else [])
                if isinstance(v, (int, float))}
        print(os.path.relpath(c, C.PROJECT_ROOT))
        for k, v in flat.items():
            if "auc" in k.lower():
                print(f"    {k}: {v}")
    except Exception:
        pass

: 

## Browse any report
Paste a path from the list above to dump its full contents + PNGs.

In [ ]:
target = reports[0] if reports else None   # <- or set a path string
if target:
    print(os.path.relpath(target, C.PROJECT_ROOT), "\n")
    print(json.dumps(C.load_json(target), indent=2)[:2000])
    subdir = os.path.relpath(os.path.dirname(target), C.RESULTS_DIR)
    C.show_pngs(C.find_pngs(subdir), ncols=2, max_imgs=6)

: 

## Browse result figures by folder
List of result subfolders, then render the PNGs in whichever you pick.

In [ ]:
subdirs = sorted(d for d in os.listdir(C.RESULTS_DIR)
                 if os.path.isdir(os.path.join(C.RESULTS_DIR, d)))
print("Result folders:", subdirs)

PICK = subdirs[0] if subdirs else ""     # <- change to any folder above
C.show_pngs(C.find_pngs(PICK), ncols=2, max_imgs=8)

: 